# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name)
print(metadata.description)
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"Data Collection Type: {getattr(metadata, 'dataCollectionType', 'N/A')}")

## 2. Data Overview
Review available record sets and their associated fields, using their `@id`s for reference.

In [ ]:
# List available record sets
record_sets = dataset.metadata.recordSet
if not record_sets:
    print('No record sets are listed directly in metadata. Attempting to infer available record sets from loaded resources...')

# mlcroissant automatically infers available record sets from data files
record_set_ids = dataset.list_record_sets()

print("Available record sets (by @id):")
for rs_id in record_set_ids:
    print(f" - {rs_id}")

# Display fields for each record set
for rs_id in record_set_ids:
    print(f"\nFields in record set {rs_id}:")
    fields = dataset.list_fields(record_set=rs_id)
    for fld in fields:
        print(f"  - Field @id: {fld}")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. 
Reference all fields by their `@id`s, as listed previously.

In [ ]:
# Choose a record set and known numeric field @id
# For demonstration, use the first record set; adjust field ids as needed for your dataset
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[example_record_set_id]

# Display available columns
print("Available columns:", df.columns.tolist())

# Pick a numeric field (e.g., 'age') by @id if present
possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()]
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else None
print(f"Using numeric field for analysis: {numeric_field}")

if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'location' in col.lower() or 'status' in col.lower() or 'anatomic' in col.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found or available for analysis.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

In [ ]:
# Visualization using matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field} in Record Set {example_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No valid numeric field for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² dataset using the Croissant schema and the `mlcroissant` library. By referencing entities via their `@id`s, you can flexibly extract and process all record sets and fields. The sample EDA shows filtering, normalization, and basic grouping, and visualizations illustrate numeric distributions.

- The dataset provides clinical and molecular variables for second primary colorectal cancer survivors.
- Using Croissant's schema and Python tools, you can quickly inspect and manipulate tabular clinical datasets.
- All steps referenced entities by their unique `@id`s, ensuring reproducibility and clarity.

For further analysis, apply advanced modeling, missing data handling, or integration with additional clinical datasets.